# Synthea Stories: generate GP notes and patient letters

This notebook turns a synthetic patient record from [Synthea](https://synthea.mitre.org/) (a C-CDA `.xml` file)
into a long, realistic text record: GP consultation notes and messages from the patient, written by an
open-weight LLM running on Colab's free GPU. **Every patient and document is fictional.**

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the cells in order.

How long it takes: setup is about 5 minutes. With `gemma3:12b` on the free T4 GPU one patient of ~45 documents
takes about 25 minutes on its own. Several patients are generated side by side (`WORKERS` below), which
gets more done per hour than running them one at a time.

**If Colab disconnects:** run steps 1–5 again, then re-run the generation step. Documents already written are
saved in Google Drive and are skipped, so it carries on where it stopped.

## 1. Check the GPU
You should see a Tesla T4 below. If you see an error, change the runtime type as described above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
Re-running this cell always fetches the latest version from GitHub.

In [ ]:
import os, sys
REPO_DIR = "/content/Synthea-Stories"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Seth-Bullock/Synthea-Stories.git {REPO_DIR}
# Always bring the code fully up to date (discarding any local edits).
!git -C {REPO_DIR} fetch -q origin main && git -C {REPO_DIR} reset -q --hard origin/main
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# Forget any older copy of the code Python already loaded in this session.
for name in [m for m in sys.modules if m == "gpnotes" or m.startswith("gpnotes.")]:
    del sys.modules[name]
!git -C {REPO_DIR} log -1 --format="Code version: %h (%cd)" --date=format:"%d %b %Y %H:%M"

## 3. Choose where to save the output
Saving to Google Drive (recommended) means nothing is lost if Colab disconnects. You will be asked to allow
access to your Drive. Set `USE_DRIVE = False` to save inside this Colab session instead (deleted when it ends).

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/synthea-stories-output"
else:
    OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output will be saved in", OUTPUT_DIR)

## 4. Install and start Ollama
[Ollama](https://ollama.com) runs open-weight LLMs locally on the GPU.

`WORKERS` is how many patients are written at the same time. The GPU can produce several responses together
for little more than the cost of one, so 3 workers should take much less than 3 times as long as 1. Each
worker needs extra GPU memory, so if step 5 shows the model is not "100% GPU", lower `WORKERS` and re-run
this cell.

In [ ]:
import subprocess, time, urllib.request

WORKERS = 3

!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

!pkill -f "ollama serve"; sleep 2   # stop any server from an earlier run of this cell
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ,
                               "OLLAMA_KEEP_ALIVE": "-1",              # keep the model loaded
                               "OLLAMA_NUM_PARALLEL": str(WORKERS),    # requests served at once
                               "OLLAMA_FLASH_ATTENTION": "1",          # faster, less memory
                               "OLLAMA_KV_CACHE_TYPE": "q8_0"})        # halves the memory used per worker
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama is running.")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start; see /content/ollama.log")

## 5. Download the model
`gemma3:12b` (Google's Gemma 3, 12 billion parameters, about 8 GB) is the largest good writer that fits
comfortably on the free T4 GPU. For a faster but weaker model, try `gemma3:4b`. Other options include
`qwen2.5:14b` and `llama3.1:8b`.

In [ ]:
MODEL = "gemma3:12b"
!ollama pull {MODEL} 2>&1 | tail -1

from gpnotes.llm import OllamaClient
client = OllamaClient(model=MODEL, num_ctx=8192, temperature=0.8)
print(client.chat("You are a helpful assistant.", "Say hello in exactly five words."))

# The PROCESSOR column should say "100% GPU". Anything that mentions CPU will be very slow.
!ollama ps

## 6. Choose the patient record(s)
Set `SOURCE` to one of:
- `"sample"`: the sample patient that comes with the code
- `"drive"`: every `.xml` file in a folder on your Google Drive (best for many patients; put the Synthea
  `ccda` files in `MyDrive/synthea-ccda`, or change `DRIVE_INPUT_DIR`)
- `"upload"`: choose files from your computer when prompted

In [ ]:
SOURCE = "sample"
DRIVE_INPUT_DIR = "/content/drive/MyDrive/synthea-ccda"

if SOURCE == "drive":
    CCDA_FILES = sorted(os.path.join(DRIVE_INPUT_DIR, n) for n in os.listdir(DRIVE_INPUT_DIR) if n.endswith(".xml"))
elif SOURCE == "upload":
    from google.colab import files
    os.makedirs("/content/input", exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join("/content/input", name), "wb") as f:
            f.write(data)
    CCDA_FILES = sorted(os.path.join("/content/input", n) for n in os.listdir("/content/input") if n.endswith(".xml"))
else:
    CCDA_FILES = ["samples/Alexandra16_Mosciski958.xml"]
print(len(CCDA_FILES), "file(s):", [os.path.basename(f) for f in CCDA_FILES[:5]], "..." if len(CCDA_FILES) > 5 else "")

## 7. Quick test: the first few episodes of one patient
This writes the first 6 episodes (about 8 documents) to a separate `test` folder so you can check the style
before committing to a full run. The exact prompts sent to the model are saved in `<patient>/prompts/`.

In [ ]:
from gpnotes.generate import generate_patient

test_file = generate_patient(CCDA_FILES[0], client, out_dir=os.path.join(OUTPUT_DIR, "test"), max_episodes=6)
print(open(test_file).read())

## 8. Full run
Generates the complete record for every chosen patient, `WORKERS` patients at a time. Safe to re-run after a
disconnection: finished documents are skipped. Messages from different patients are interleaved; each line
starts with the patient's number.

`MAX_DOCUMENTS` caps the documents per patient. Synthea records vary enormously (one elderly patient can have
hundreds of visits); with a cap, the most eventful visits are kept and routine repeat visits are dropped.
Set it to `None` for no cap.

In [ ]:
from gpnotes.generate import generate_many

MAX_DOCUMENTS = 60

outputs = generate_many(CCDA_FILES, client, out_dir=OUTPUT_DIR, workers=WORKERS, max_documents=MAX_DOCUMENTS)

## 9. Read and download the results

In [ ]:
from google.colab import files

print(open(outputs[0]).read()[:5000])

# The finished records are already in your Drive. To download them all as one zip file:
import shutil
zip_path = shutil.make_archive("/content/synthea_stories", "zip", root_dir=OUTPUT_DIR, base_dir=".")
files.download(zip_path)